# 02 — Metric & Seasonal-Naive Baseline

**The bar the model has to clear.**

Forecasting is uniquely easy to fool yourself with. The defence is to fix the
metric and build the naive baseline *before* touching a model, then report every
later result against it on identical test rows.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)


In [ ]:
from src.forecast import build_supervised, seasonal_naive, wape, mape, bias

weekly = pd.read_csv(C.WEEKLY_PATH, parse_dates=["week_start"])
features = pd.read_csv(C.FEATURES_PATH, parse_dates=["week_start", "launch_date"])
print(f"{weekly['sku_id'].nunique()} SKUs, {weekly['week_start'].nunique()} weeks")

## 1. The metric

**WAPE** = Σ|actual − forecast| ÷ Σ|actual|.

Chosen over MAPE because notebook 01 showed ~4.5% of SKU-weeks have zero demand,
where MAPE is undefined. WAPE also weights by volume, so being wrong about a
best-seller costs more than being wrong about a slow mover — which is exactly how
NorthBay experiences the error.

**Bias** (signed error) is tracked alongside it. A model can have decent WAPE
while systematically under-ordering, which would quietly cause stockouts.

In [ ]:
a = np.array([10.0, 20.0, 30.0])
print("perfect forecast     WAPE:", wape(a, a))
print("10% over everywhere  WAPE:", round(wape(a, a * 1.1), 4), " bias:", round(bias(a, a * 1.1), 4))
print("10% under everywhere WAPE:", round(wape(a, a * 0.9), 4), " bias:", round(bias(a, a * 0.9), 4))

## 2. The horizon

8 weeks. Long enough to cover the longest replenishment lead time in the
inventory data, short enough that the forecast is still informative.

In [ ]:
inv = pd.read_csv(C.INVENTORY_PATH)
print(inv["lead_time_days"].describe())
print(f"\nLongest lead time: {inv['lead_time_days'].max():.0f} days "
      f"= {inv['lead_time_days'].max() / 7:.1f} weeks")
print(f"Chosen horizon: {C.HORIZON_WEEKS} weeks")

## 3. The baseline

Seasonal-naive: demand in the target week equals demand in the same week one
year earlier. Where that week does not exist (young SKUs), fall back to the mean
of the last 4 observed weeks — the honest fallback a human planner would use.

In [ ]:
from src.pipeline import encode_categories

sup = build_supervised(encode_categories(features))
labelled = sup[sup["y"].notna()].copy()

last_week = labelled["target_week"].max()
cutoff = last_week - pd.Timedelta(weeks=C.HORIZON_WEEKS)
test = labelled[(labelled["target_week"] >= cutoff) & (labelled["origin_week"] == cutoff)]

base = seasonal_naive(test, weekly)
actual = test["y"].to_numpy()

print(f"Test rows: {len(test)}")
print(f"Baseline WAPE : {wape(actual, base):.4f}")
print(f"Baseline bias : {bias(actual, base):+.4f}")

## 4. Where the baseline fails

Knowing *which* SKUs the naive forecast handles badly tells us what the model
has to add. If the baseline were uniformly good, there would be little to gain.

In [ ]:
t = test.assign(base=base, err=np.abs(actual - base))
per_sku = t.groupby("sku_id").agg(actual=("y", "sum"), err=("err", "sum"))
per_sku["wape"] = per_sku["err"] / per_sku["actual"].replace(0, np.nan)

print("Worst 10 SKUs for the baseline:")
print(per_sku.sort_values("wape", ascending=False).head(10).round(3))
print("\nBaseline WAPE by demand size:")
per_sku["bucket"] = pd.qcut(per_sku["actual"], 4, labels=["lowest", "low", "high", "highest"])
print(per_sku.groupby("bucket", observed=True)["wape"].median().round(3))

**Read-through:** the baseline is weakest on low-volume and volatile SKUs,
which is where pooling information across products should help most. That is the
argument for a single global model rather than 200 per-SKU models.